# SCMF-QAOA 小规模论文复现

本 notebook 复现 *Self-consistent mean-field quantum approximate optimization*（arXiv:2603.09838v1）的第一阶段链路。它只负责生成问题、调用 solver、展示指标和做简单断言；Gaussian SK 定义与 SCMF-QAOA 逻辑均位于脚本中。

## 1. 环境与复现范围

当前范围是理想 statevector、$p=1$、8-spin Gaussian SK。对照包括完整穷举、完整 QAOA，以及关闭 environment 更新的独立子问题基线。

In [ ]:
import copy
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "lib").is_dir() or not (PROJECT_ROOT / "problem").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the QSolutionData repository root.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lib.contracts import validate_qubo, validate_qubo_result
from lib.solvers.qubo import ExactQuboSolver, QaoaQuboSolver, ScmfQaoaSolver
from problem.reproductions import build_scmf_gaussian_sk_instance

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

## 2. 论文一致的 Gaussian SK 问题

生成器输出 `qubo.v1`，同时在 metadata 中保留零场、完全图和未缩放 Gaussian coupling。

In [ ]:
problem = build_scmf_gaussian_sk_instance(spin_count=8, seed=2603)
problem_snapshot = copy.deepcopy(problem)
validate_qubo(problem)

generator = problem["metadata"]["generator"]
print("Problem:", problem["problem_id"])
print("Variables:", problem["num_variables"])
print("Couplings:", len(problem["metadata"]["paper_ising"]["couplings"]))
print("Distribution:", generator["parameters"]["coupling_distribution"])
print("Normalization:", generator["parameters"]["normalization"])

## 3. Exact 与完整 QAOA 对照

Exact 提供真实 ground-state energy；普通 QAOA 使用完整 8-qubit statevector。启发式即使命中最优样本，状态仍为 `feasible`。

In [ ]:
exact_result = ExactQuboSolver().solve(problem, {"max_variables": 12})
qaoa_result = QaoaQuboSolver().solve(
    problem,
    {
        "layers": 1,
        "optimizer_iterations": 12,
        "restarts": 3,
        "shots": 512,
        "seed": 7,
        "max_variables": 12,
    },
)
validate_qubo_result(problem, exact_result)
validate_qubo_result(problem, qaoa_result)

print("Exact energy:", exact_result["best_energy"])
print("QAOA energy:", qaoa_result["best_energy"])
print("QAOA expectation:", qaoa_result["metrics"]["expectation"])

## 4. Environmentless 与 self-consistent decomposition

两次运行使用相同问题、分区、外层预算和采样预算。唯一关键区别是是否执行 environment self-consistency。

In [ ]:
shared_config = {
    "layers": 1,
    "subproblem_count": 2,
    "optimizer_iterations": 8,
    "shots": 512,
    "seed": 7,
    "max_subproblem_variables": 8,
}

independent_result = ScmfQaoaSolver().solve(
    problem,
    {**shared_config, "max_environment_sweeps": 0},
)
scmf_result = ScmfQaoaSolver().solve(
    problem,
    {
        **shared_config,
        "max_environment_sweeps": 60,
        "environment_tolerance": 1e-5,
        "energy_tolerance": 1e-5,
    },
)
scmf_repeat = ScmfQaoaSolver().solve(
    problem,
    {
        **shared_config,
        "max_environment_sweeps": 60,
        "environment_tolerance": 1e-5,
        "energy_tolerance": 1e-5,
    },
)

validate_qubo_result(problem, independent_result)
validate_qubo_result(problem, scmf_result)

## 5. 结果与论文机制检查

In [ ]:
comparison = [
    {"solver": "Exact", "energy": exact_result["best_energy"], "expectation": None},
    {"solver": "full QAOA", "energy": qaoa_result["best_energy"], "expectation": qaoa_result["metrics"]["expectation"]},
    {"solver": "independent subproblems", "energy": independent_result["best_energy"], "expectation": independent_result["metrics"]["expectation"]},
    {"solver": "SCMF-QAOA", "energy": scmf_result["best_energy"], "expectation": scmf_result["metrics"]["expectation"]},
]

for row in comparison:
    print(row)

print("Partitions:", scmf_result["metadata"]["partitions"])
print("Fixed spin witness:", scmf_result["metadata"]["fixed_spins"])
print("Environment sweeps:", scmf_result["metrics"]["environment_sweeps"])
print("Environment converged:", scmf_result["metrics"]["environment_converged"])

In [ ]:
assert exact_result["status"] == "optimal"
assert qaoa_result["status"] == "feasible"
assert independent_result["termination_reason"] == "independent_subproblems_completed"
assert scmf_result["status"] == "feasible"
assert scmf_result["metrics"]["environment_converged"] is True
assert scmf_result["metrics"]["expectation"] < independent_result["metrics"]["expectation"]
assert scmf_result["metadata"]["fixed_spins"] == [[7, 1]]
assert scmf_result["best_sample"][7] == 0
assert scmf_result["best_energy"] >= exact_result["best_energy"] - 1e-10
assert scmf_result["best_sample"] == scmf_repeat["best_sample"]
assert scmf_result["metrics"] == scmf_repeat["metrics"]
assert scmf_result["metadata"] == scmf_repeat["metadata"]
assert problem == problem_snapshot

print("SCMF-QAOA reproduction checks passed.")

## 结论

该微型复现确认了 Gaussian SK → QUBO、零场 symmetry breaking、共享角度、environment self-consistency、product-state sampling 和 canonical result 的完整链路。它不代表对论文大规模曲线、分子对接或 Rigetti 硬件实验的复刻。